# Basin data generation at scale — variety grammar + vLLM

The scaled-up version of `12_basin_lab.ipynb`:

1. **Variety** — `basin_variety.py` samples N prompts from a ~100k-prompt grammar
   (10 families × templates × actors × slots × reflection stems; `goodnews` = positive control)
2. **Corpus** — vLLM, 1500 prompts × 8 samples × 3 organisms = 36k rollouts (~minutes per organism)
3. **Cluster** — embed endpoints, HDBSCAN
4. **Transitions** — resample 8 continuations at depths 12/24/36/48 (subset of 400 prompts × 2 rollouts)
5. **Metastability** — merge exchanging clusters → final basin map + commitment curves

One organism per subprocess (clean GPU teardown between vLLM models). A100 recommended.

In [ ]:
import os
if not os.path.exists("/content/dt_rl"):
    !git clone https://github.com/ChuloIva/dt_rl.git /content/dt_rl
%cd /content/dt_rl
!git pull
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception as e:
    print("no HF_TOKEN from userdata:", e)
%pip install -q -U vllm sentence-transformers scikit-learn

## 1 — generate the prompt set from the variety grammar

1500 sampled prompts + 200 matched visibility triples (private/public/glory —
the Wallace & Baumeister self-enhancement-opportunity knob) = 2100 prompts.


In [ ]:
!python scripts/basin_variety.py --n 1500 --glory-n 200 --seed 0 --out data/basin_prompts_gen.json
import json, random
ps = json.load(open("data/basin_prompts_gen.json"))["prompts"]
for p in random.Random(0).sample(ps, 8):
    print(f"[{p['family']:<11}] {p['prompt']}")

## 2 — rollout corpus (vLLM, one organism per subprocess)

In [ ]:
for org in ["base", "dark", "clinical-depression"]:
    !python scripts/basin_corpus_vllm.py --organism {org} \
        --prompts data/basin_prompts_gen.json --out data/basin_corpus_xl
!wc -l data/basin_corpus_xl/rollouts_*.jsonl

## 3 — cluster the 36k endpoints

In [ ]:
!python scripts/basin_cluster.py --corpus data/basin_corpus_xl --min-cluster-size 60
from IPython.display import Markdown, Image, display
display(Image("data/basin_corpus_xl/clusters/scatter.png"))
display(Markdown(open("data/basin_corpus_xl/clusters/report.md").read()))

## 4 — perturb & resample

400 prompts × 2 rollouts × 4 depths × 8 resamples per organism (~76k short generations total).

In [ ]:
for org in ["base", "dark", "clinical-depression"]:
    !python scripts/basin_transitions_vllm.py --organism {org} --corpus data/basin_corpus_xl
!python scripts/basin_transitions_vllm.py --classify --corpus data/basin_corpus_xl
!wc -l data/basin_corpus_xl/transitions/transitions.jsonl

## 5 — metastability: the basin map

In [ ]:
!python scripts/basin_metastability.py --corpus data/basin_corpus_xl
from IPython.display import Markdown, Image, display
display(Image("data/basin_corpus_xl/basins/commitment.png"))
display(Markdown(open("data/basin_corpus_xl/basins/report.md").read()))

## 6 — save

Token ids in the corpus are what Build 4's value head will teacher-force — keep everything.

In [ ]:
!zip -qr basin_xl_results.zip data/basin_corpus_xl
!ls -lh basin_xl_results.zip
# from google.colab import files; files.download("basin_xl_results.zip")
# or: from notebooks.colab_setup import mount_drive; d = mount_drive(); !cp basin_xl_results.zip {d}/results/